<a href="https://colab.research.google.com/github/ashleyt-serene/ashley/blob/main/newtitlesSalesPrediction_rev7Hist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#パッケージの準備
import pandas as pd
import datetime
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import export_graphviz
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import graphviz

In [ ]:
#random forest
from sklearn.ensemble import RandomForestRegressor
# 可視化に必要なライブラリをインポート

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# 可視化の際に見やすくする
from sklearn.ensemble import RandomForestRegressor


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import warnings
import os

In [ ]:
# Get today's date
today = datetime.date.today()

# Format the date as a string
date_string = today.strftime("%Y%m%d")

# Specify the specific file name
input_file_name = "/content/drive/MyDrive/Colab Notebooks/"
output_file_name = "/content/drive/MyDrive/Colab Notebooks/predict_Title_List"
output_ModelReview = "/content/drive/MyDrive/Colab Notebooks/ModelReview"
output_pkl = "/content/drive/MyDrive/Colab Notebooks/pkl"


# Concatenate the formatted date with the specific file name to create the file path
inputfile_path = f"{input_file_name}{date_string}.xlsx"
outputfile_path = f"{output_file_name}_{date_string}.xlsx"
pkl_outputfile_path = f"{output_pkl}_{date_string}.pkl"
ModelReview_outputfile_path = f"{output_ModelReview}_{date_string}.xlsx"

print("Excel input file path:", inputfile_path)
print("Excel output file path:", outputfile_path)
print("PKL output file path:", pkl_outputfile_path)
print("ModelReview_outputfile_path:", ModelReview_outputfile_path)

In [ ]:
#学習用のデータを投入
newtitles = pd.read_excel(inputfile_path)


In [ ]:
newtitles.head()

In [ ]:
len(newtitles)

In [ ]:
newtitles['GENRENAME'].unique()

In [ ]:
newtitles = newtitles.fillna(0)

In [ ]:
newtitles.info()

In [ ]:
#リリース年を作成する
from typing_extensions import NewType
newtitles['started_sale_at']= pd.to_datetime(newtitles['started_sale_at'])
newtitles['ReleasedYear'] = newtitles['started_sale_at'].dt.year
print(newtitles.dtypes)


In [ ]:
#初回更新話をカテゴリタイプとしてカラムの作成後、ダミー変数化する
newtitles['epcntRange']= pd.cut(newtitles['maxep'], bins=[-1,19,22,423],labels = ['less20','btw20to22','morethan22'])



In [ ]:
from sklearn.preprocessing import LabelEncoder

# インスタンスの作成
label_encoders = {}

# epcntRangeのエンコード
label_encoders['epcntRange'] = LabelEncoder()
newtitles['epcntRange'] = label_encoders['epcntRange'].fit_transform(newtitles['epcntRange'])

# GENRENAMEのエンコード
label_encoders['GENRENAME'] = LabelEncoder()
newtitles['GENRENAME'] = label_encoders['GENRENAME'].fit_transform(newtitles['GENRENAME'])

# GenderTargetのエンコード
label_encoders['GenderTarget'] = LabelEncoder()
newtitles['GenderTarget'] = label_encoders['GenderTarget'].fit_transform(newtitles['GenderTarget'])

In [ ]:
# 条件の設定
cond0 = newtitles['started_sale_at'] >= pd.to_datetime('today') - pd.DateOffset(months=26)
cond1 = newtitles['started_sale_at'] <= pd.to_datetime('today') - pd.DateOffset(months=2)
cond2 = newtitles['LastfromRelease'] >= 60

# フィルタリング
train = newtitles[cond0 & cond1 & cond2]

# 結果の表示
len(train)

In [ ]:
cols = ['GENRENAME', 'epcntRange','GenderTarget', 'day1total', 'day1ru','day1pru', 'ep5pru', 'ep7pru', 'ep9pru', 'ep11pru', 'ep12pru', 'ep13pru', 'ep14pru', 'ep15pru','ep16pru', 'ep17pru', 'ep18pru', 'ep19pru', 'ep20pru','ReleasedYear']


In [ ]:
#다중공선성의 체크

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 데이터프레임에서 설명변수만 선택
X = train[cols]  # 예시로 설명변수가 x1, x2, x3, x4라고 가정

# 상수항(intercept) 추가
X = sm.add_constant(X)

# VIF 계산
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

In [ ]:
train.info()

In [ ]:
#目的変数（３１〜60日間のt売上）の分離
y_train = train.pop("day60total")


In [ ]:
y_train.isnull().sum()

In [ ]:
#説明変数の分離

cols = ['GENRENAME', 'epcntRange','GenderTarget', 'day1total', 'day1ru','day1pru', 'ep5pru', 'ep7pru', 'ep9pru', 'ep11pru', 'ep12pru', 'ep13pru', 'ep14pru', 'ep15pru','ep16pru', 'ep17pru', 'ep18pru', 'ep19pru', 'ep20pru','ReleasedYear']
train = train[cols]

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestRegressor

# rf モデルのインスタンス化
rf = RandomForestRegressor(random_state=0, max_depth=5, min_samples_leaf=8, min_samples_split=8, n_estimators=200)

# 交差検証の実行 (R²スコア)
score_r2 = cross_validate(rf, train, y_train, cv=5, scoring='r2', return_train_score=True)

# 交差検証の実行 (負の平均二乗誤差)
score_mse = cross_validate(rf, train, y_train, cv=5, scoring='neg_mean_squared_error', return_train_score=True)

# 結果の表示
print("R² scores for each fold: ", score_r2['test_score'])
print("Mean R² score: ", score_r2['test_score'].mean())
print("Negative Mean Squared Error scores for each fold: ", score_mse['test_score'])
print("Mean Negative Mean Squared Error score: ", score_mse['test_score'].mean())

In [ ]:
cross_validate(rf, train, y_train, cv=5, scoring='neg_mean_squared_error', return_train_score=True)


In [ ]:

# RMSEを保存するための辞書を作成
rmse_by_year = {}

# 各年度ごとにモデルを訓練し、RMSEを計算
for year in train['ReleasedYear'].unique():
    train_year = train[train['ReleasedYear'] == year]
    y_train_year = y_train[train['ReleasedYear'] == year]

    # 交差検証を使用してRMSEを計算
    score = cross_validate(rf, train_year, y_train_year, cv=5, scoring='neg_mean_squared_error', return_train_score=True)

    # RMSEスコアの計算
    train_rmse = np.sqrt(-score['train_score'])
    test_rmse = np.sqrt(-score['test_score'])

    # 平均RMSEの計算
    mean_train_rmse = np.mean(train_rmse)
    mean_test_rmse = np.mean(test_rmse)

    rmse_by_year[year] = {
        'Mean Train RMSE': mean_train_rmse,
        'Mean Test RMSE': mean_test_rmse
    }

# RMSEをデータフレームに変換
rmse_df = pd.DataFrame.from_dict(rmse_by_year, orient='index').reset_index()
rmse_df.rename(columns={'index': 'ReleasedYear'}, inplace=True)


In [ ]:
rf.fit(train,y_train)
pred=rf.predict(train)
pred

In [ ]:
#モデルが提示する重要度の確認

feature_importances = pd.DataFrame([train.columns, rf.feature_importances_]).T
feature_importances.columns = ['features', 'importances']
feature_importances

In [ ]:
#重要度の保存

feature_importances = feature_importances.sort_values(by='importances', ascending=False)


In [ ]:
#重要度の可視化

plt.figure(figsize=(20,10))
plt.title('Importances')
plt.rcParams['font.size']=10
fig = sns.barplot(y=feature_importances['features'], x=feature_importances['importances'], palette='viridis')


In [ ]:
#モデルの保存
import joblib
import pickle
# モデルの保存
joblib.dump(rf, pkl_outputfile_path)



In [ ]:
# モデルのロード
loaded_model = joblib.load(pkl_outputfile_path)



In [ ]:
expect = newtitles

In [ ]:
# 条件の設定
cond0 = newtitles['started_sale_at'] >= pd.to_datetime('today') - pd.DateOffset(months=24)

# フィルタリング
expect = newtitles[cond0]


In [ ]:
expect.sort_values(by='started_sale_at', ascending=False)


In [ ]:
#モデル実行用input の準備
expect_copy = expect[cols]

In [ ]:

# 存在しないカラムをデフォルト値（例: 0）で追加する
for col in cols:
    if col not in expect_copy.columns:
        expect_copy[cols] = 0

# 必要なカラムが存在するか確認
valid_cols = [col for col in cols if col in expect_copy.columns]
expect_copy = expect[valid_cols]



In [ ]:
# expect_copy

In [ ]:
#予測実行
expect['predict'] = loaded_model.predict(expect_copy)

expect = expect.sort_values(by='predict', ascending=False)


In [ ]:
len(expect)

In [ ]:

# 元のカテゴリカルデータに戻す（デコード）
expect['epcntRange'] = label_encoders['epcntRange'].inverse_transform(expect['epcntRange'])
expect['GENRENAME'] = label_encoders['GENRENAME'].inverse_transform(expect['GENRENAME'])
expect['GenderTarget'] = label_encoders['GenderTarget'].inverse_transform(expect['GenderTarget'])

In [ ]:
# カスタム関数を定義
def determine_sales_range(total_amt_m2):
    if total_amt_m2 >= 3000000 and total_amt_m2 < 5000000:
        return 'C:300万'
    elif total_amt_m2 >= 5000000 and total_amt_m2 < 10000000:
        return 'B:500万'
    elif total_amt_m2 >= 2000000 and total_amt_m2 < 3000000:
        return 'D:200万'
    elif total_amt_m2 >= 10000000 and total_amt_m2 < 30000000:
        return 'A:1000万~3000万未満'
    elif total_amt_m2 >= 30000000:
        return 'S:3000万~'
    elif total_amt_m2 < 2000000:
        return 'E:~200万'
    else:
        return '対象外'

# # `total_amt_m2`に基づいて`salesRange`カラムを追加
# Apply the function only where cond2 is True

expect.loc[cond2, 'salesRange_total_amt_m2'] = expect.loc[cond2, 'day60total'].apply(determine_sales_range)

expect['salesRange_Prediction'] = expect['predict'].apply(determine_sales_range)


# `Prediction`に基づいて`salesRange`カラムを追加
expect['salesRange_Prediction'] = expect['predict'].apply(determine_sales_range)



In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error


In [ ]:
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
#forレビュー
review = expect[cond2]

In [ ]:

# RMSEの計算関数
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# 年度別のサンプル件数をカウント
sample_counts = review.groupby('ReleasedYear').size()

# 年度別のRMSEを計算
rmse_values = review.groupby('ReleasedYear').apply(lambda group: calculate_rmse(group['day60total'], group['predict']))


# 結果をデータフレームにまとめる
results = pd.DataFrame({
    'Sample Count': sample_counts,
    'RMSE': rmse_values
})

In [ ]:
# ReleasedYear['salesRange_total_amt_m2']グループ化して結果を計算
results_by_year_and_sale_range = review.groupby(['ReleasedYear', 'salesRange_total_amt_m2']).apply(lambda group: pd.Series({
    'Sample Count': len(group),
    'RMSE': calculate_rmse(group['day60total'], group['predict'])
})).unstack(level=0)

# 年度別のトータル行を計算して追加
total_by_year = review.groupby('ReleasedYear').apply(lambda group: pd.Series({
    'Sample Count': len(group),
    'RMSE': calculate_rmse(group['day60total'], group['predict'])
}))

# トータル行を追加
total_by_year.index = [f"Total {year}" for year in total_by_year.index]
results_with_totals = pd.concat([results_by_year_and_sale_range, total_by_year])


In [ ]:

# ReleasedYearとGenderTargetでグループ化して結果を計算
results_by_year_and_gender = review.groupby(['ReleasedYear', 'GenderTarget']).apply(lambda group: pd.Series({
    'Sample Count': len(group),
    'RMSE': calculate_rmse(group['day60total'], group['predict'])
})).unstack(level=0)

# ReleasedYearとGenderTargetとSaleRangePredictでグループ化して結果を計算
results_by_year_gender_sale_range = review.groupby(['ReleasedYear', 'GenderTarget', 'salesRange_total_amt_m2']).apply(lambda group: pd.Series({
    'Sample Count': len(group),
    'RMSE': calculate_rmse(group['day60total'], group['predict'])
})).unstack(level=0)

# 結果を1つのExcelファイルに保存
with pd.ExcelWriter(ModelReview_outputfile_path) as writer:
    rmse_df.to_excel(writer, sheet_name=' rmse_df')
    feature_importances.to_excel(writer, sheet_name='feature_importances')
    results_with_totals.to_excel(writer, sheet_name='Year_SaleRange')
    results_by_year_and_gender.to_excel(writer, sheet_name='Year_Gender')
    results_by_year_gender_sale_range.to_excel(writer, sheet_name='Year_Gender_SaleRange')

In [ ]:

# # 削除するカラムが存在するか確認
# drop_col = [
#     'genre_グルメ', 'genre_スポーツ', 'genre_ドラマ', 'genre_ファンタジー', 'genre_ホラー・ミステリー',
#     'genre_恋愛', 'genre_日常', 'genre_裏社会・アングラ', 'ep5pru', 'ep7pru', 'ep9pru', 'ep11pru',
#     'ep13pru', 'ep14pru', 'ep15pru', 'ep16pru', 'ep17pru', 'ep18pru', 'ep19pru', 'EP_btw20to22', 'EP_morethan22'
# ]

# # 存在するカラムのみ削除
# existing_drop_cols = [col for col in drop_col if col in expect.columns]
# expect = expect.drop(existing_drop_cols, axis=1)

# # 再度ソート
expect = expect.sort_values(by='predict', ascending=False)

# 新しいカラムを計算
expect['day1prur'] = expect['day1pru'] / expect['day1ru']
expect['12ep_retrate'] = expect['ep12pru'] / expect['day1pru']
expect['20ep_retrate'] = expect['ep20pru'] / expect['day1pru']

In [ ]:

# 予測ファイルの保存
expect.to_excel(outputfile_path, index=False)

In [ ]:
len(expect)